# AJSR revision analyses (reviewer comments 3, 4, 6, 7)

Uses the same Drive folders as `AI_UrbanPlanning.ipynb`:
* `MyDrive/AI-UrbanPlanning/singapore_tiles.csv` and `.gpkg` (script 01 outputs)
* `MyDrive/dataset.zip` (script 04 output: manifest + images)
* `MyDrive/pix2pix_runs/` (checkpoints, `evaluation/sweep_metrics.csv`, `evaluation/baseline_native_metrics.csv`, and the `fid_*` image folders)

**Run everything with Runtime > Run all.** A CPU runtime is fine, and nothing is retrained. Results go to `MyDrive/pix2pix_runs/revision_results/`, and the last cell zips them to `MyDrive/revision_results_upload.zip`. Upload that zip to the chat.

It does not modify any of your existing files.

In [ ]:
# 1. Mount Drive and check every input exists
from google.colab import drive
drive.mount('/content/drive')
import os
MY = '/content/drive/MyDrive'
TILES_CSV  = f'{MY}/AI-UrbanPlanning/singapore_tiles.csv'
TILES_GPKG = f'{MY}/AI-UrbanPlanning/singapore_tiles.gpkg'
DATASET_ZIP = f'{MY}/dataset.zip'
RUNS = f'{MY}/pix2pix_runs'; EVAL = f'{RUNS}/evaluation'
OUT = f'{RUNS}/revision_results'; os.makedirs(OUT, exist_ok=True)
need = {'tiles csv': TILES_CSV, 'tiles gpkg': TILES_GPKG, 'dataset.zip': DATASET_ZIP,
        'sweep_metrics.csv': f'{EVAL}/sweep_metrics.csv',
        'baseline_native_metrics.csv': f'{EVAL}/baseline_native_metrics.csv',
        'fid_real/': f'{EVAL}/fid_real', 'fid_cond/': f'{EVAL}/fid_cond',
        'fid_cond_shuffled/': f'{EVAL}/fid_cond_shuffled', 'fid_base_zero/': f'{EVAL}/fid_base_zero'}
for k, p in need.items():
    print(('OK      ' if os.path.exists(p) else 'MISSING ') + k.ljust(30) + p)
print('\nOther files in pix2pix_runs:', sorted(os.listdir(RUNS)))
print('Files in AI-UrbanPlanning:', sorted(os.listdir(f'{MY}/AI-UrbanPlanning')))

In [ ]:
# 2. Install packages and unpack the dataset
!pip -q install geopandas statsmodels pytorch-fid
import zipfile, shutil
if not os.path.exists('/content/dataset/manifest.csv'):
    zipfile.ZipFile(DATASET_ZIP).extractall('/content/dataset')
print(os.listdir('/content/dataset'))
# local copies of the evaluation image folders (much faster than reading from Drive)
for d in ['fid_real', 'fid_cond', 'fid_cond_shuffled', 'fid_base_zero']:
    if os.path.isdir(f'{EVAL}/{d}') and not os.path.isdir(f'/content/{d}'):
        shutil.copytree(f'{EVAL}/{d}', f'/content/{d}')
    print(d, len(os.listdir(f'/content/{d}')) if os.path.isdir(f'/content/{d}') else 'missing')

In [ ]:
# 3. Build the analysis tables from your files (tiles, superblocks, ambiguity, sweep, baseline)
import numpy as np, pandas as pd, geopandas as gpd
from PIL import Image
t = pd.read_csv(TILES_CSV)
g = gpd.read_file(TILES_GPKG)[['tile_id', 'geometry']]
man = pd.read_csv('/content/dataset/manifest.csv')
print('singapore_tiles.csv columns:', list(t.columns))
print('land tiles in the grid (max tile_id + 1, used for standardization in script 01):', int(t.tile_id.max()) + 1)
print('manifest.csv columns:', list(man.columns))
open(f'{OUT}/columns.txt', 'w').write(f'tiles: {list(t.columns)}\nmanifest: {list(man.columns)}\nland tiles in grid: {int(t.tile_id.max()) + 1}\n')

def pick(cands):
    for c in cands:
        if c in t.columns: return c
    for c in t.columns:
        if any(k in c.lower() for k in cands): return c
    return None
PERM = pick(['permeable_ratio', 'permeable', 'perm'])
SETB = pick(['setback_m', 'mean_setback', 'setback', 'dist_to_water', 'water_dist'])
WATR = pick(['water_area', 'water_ratio', 'water_frac', 'water_share', 'water'])
print('flood components found -> permeable:', PERM, '| setback:', SETB, '| water area:', WATR)

floor = (t.n_amenities == 0) & (t.n_intersections == 0) & (t.landuse_entropy == 0)
tiles = pd.DataFrame({'tile_id': t.tile_id,
                      'intersection_density': t.n_intersections / 0.25,   # per km2 (500 m tile = 0.25 km2)
                      'amenity_density': t.n_amenities / 0.25,
                      'landuse_entropy': t.landuse_entropy,
                      'building_coverage': t.density,
                      'excluded': floor.values,
                      'walk_score': t.walkability, 'flood_score': t.flood_resilience})
if PERM: tiles['permeable_ratio'] = t[PERM]
if SETB: tiles['setback_m'] = t[SETB]
m = man[['tile_id', 'split', 'walkability_label', 'flood_resilience_label']].rename(
    columns={'walkability_label': 'walk_label', 'flood_resilience_label': 'flood_label'})
tiles = tiles.merge(m, on='tile_id', how='left')
print(f"{len(tiles)} tiles | excluded {int(tiles.excluded.sum())} | train {(tiles.split=='train').sum()} | test {(tiles.split=='test').sum()}")

# superblocks exactly as script 04 (4x4 tiles, origin = bounds of the 1,754 kept tiles)
g = g.merge(tiles[['tile_id']], on='tile_id')
kept_g = g[g.tile_id.isin(tiles.loc[~tiles.excluded, 'tile_id'])]
minx, miny = kept_g.total_bounds[0], kept_g.total_bounds[1]
cen = g.geometry.centroid
g['x'], g['y'] = cen.x, cen.y
g['superblock_id'] = (((g.x - minx) // 500).astype(int) // 4).astype(str) + '_' + (((g.y - miny) // 500).astype(int) // 4).astype(str)
tiles = tiles.merge(g[['tile_id', 'x', 'y', 'superblock_id']], on='tile_id', how='left')
k = tiles[~tiles.excluded]
mixed = k.groupby('superblock_id')['split'].nunique().gt(1).sum()
print(f"superblocks among kept tiles: {k.superblock_id.nunique()} | test superblocks: {k[k.split=='test'].superblock_id.nunique()} | superblocks with mixed split (should be 0): {mixed}")

# reproduce the split selection of script 04 (seed 42) as a check
meta = pd.read_csv(f'{MY}/singapore_tiles_filtered.csv') if os.path.exists(f'{MY}/singapore_tiles_filtered.csv') else None
if meta is not None:
    gg = gpd.read_file(TILES_GPKG)[['tile_id', 'geometry']].merge(meta[['tile_id']], on='tile_id', how='inner')
    mx, my = gg.total_bounds[0], gg.total_bounds[1]
    sb = (((gg.geometry.centroid.x - mx) // 500).astype(int) // 4).astype(str) + '_' + (((gg.geometry.centroid.y - my) // 500).astype(int) // 4).astype(str)
    rng = np.random.default_rng(42); blocks = sb.unique(); rng.shuffle(blocks)
    tb, cnt, target = set(), 0, int(len(gg) * 0.15)
    for b in blocks:
        if cnt >= target: break
        tb.add(b); cnt += (sb == b).sum()
    rep = pd.Series(np.where(sb.isin(tb), 'test', 'train'), index=gg.tile_id)
    same = (rep.reindex(man.tile_id).values == man.split.values).mean()
    print(f'split reproduced from seed 42: {same:.1%} of tiles match the manifest')
    open(f'{OUT}/split_check.txt', 'w').write(f'test superblocks {len(tb)}; reproduced {same:.4f}\n')

tiles.to_csv('/content/std_tiles.csv', index=False)

# canvas ambiguity (model independent): K nearest TRAINING tiles in input-visible features
def png_share(path, colors):
    a = np.asarray(Image.open(path).convert('RGB')).reshape(-1, 3)
    return np.mean([np.all(a == c, axis=1) for c in colors], axis=0).sum()
kt = tiles[~tiles.excluded].copy()
if WATR:
    kt['f_water'] = kt.tile_id.map(t.set_index('tile_id')[WATR])
else:  # water share of the input canvas
    kt['f_water'] = [png_share(f"/content/dataset/{s}/input/tile_{i}.png", [(0, 90, 255)]) for i, s in zip(kt.tile_id, kt.split)]
kt['f_int'] = kt.tile_id.map(t.set_index('tile_id').n_intersections)
kt['o_perm'] = kt['permeable_ratio'] if PERM else [png_share(f"/content/dataset/{s}/target/tile_{i}.png", [(0, 170, 0), (0, 90, 255)]) for i, s in zip(kt.tile_id, kt.split)]
tr, te = kt[kt.split == 'train'], kt[kt.split == 'test']
mu, sd = tr[['f_int', 'f_water']].mean(), tr[['f_int', 'f_water']].std()
Xtr = ((tr[['f_int', 'f_water']] - mu) / sd).values; Xte = ((te[['f_int', 'f_water']] - mu) / sd).values
outs = ['building_coverage', 'o_perm', 'landuse_entropy']; s_m = tr[outs].std()
amb = {}
for K in [10, 15, 25, 40, 60]:
    d = ((Xte[:, None, :] - Xtr[None, :, :]) ** 2).sum(-1)
    nn_idx = np.argsort(d, axis=1)[:, :K]
    vals = np.stack([tr[o].values[nn_idx].std(axis=1, ddof=1) / s_m[o] for o in outs], 1).mean(1)
    amb[K] = vals
canv = pd.DataFrame({'canvas_id': te.tile_id.values, 'superblock_id': te.superblock_id.values, 'ambiguity': amb[25]})
for K, v in amb.items(): canv[f'ambiguity_K{K}'] = v
canv.to_csv('/content/std_canvases.csv', index=False); canv.to_csv(f'{OUT}/canvas_ambiguity.csv', index=False)
print('ambiguity computed for', len(canv), 'held-out canvases', '(water feature from', WATR or 'input canvas pixels', ')')

sw = pd.read_csv(f'{EVAL}/sweep_metrics.csv')
sw = sw[sw.model == 'conditional'].rename(columns={'tile': 'canvas_id', 'walk': 'walk_label', 'flood': 'flood_label'})
sw.to_csv('/content/std_sweep.csv', index=False)
bs = pd.read_csv(f'{EVAL}/baseline_native_metrics.csv').rename(columns={'tile': 'canvas_id'})
bs.to_csv('/content/std_baseline.csv', index=False)
print(f'sweep rows (conditional): {len(sw)} | baseline canvases: {len(bs)}')

## 4. Write the analysis scripts (just run)

In [ ]:
%%writefile /content/r1_preprocessing_audit.py
"""
R1. Preprocessing audit and exclusion sensitivity (reviewer comments 3 and 4).

CPU only. No retraining.

What it does
  1. Recomputes the walkability (W) and flood resilient design (F) composites three ways:
       a. "as published": using the standardization the paper reports (edit PUBLISHED below)
       b. "training only": mean and SD estimated on the 1,485 training tiles only
       c. "all developed": mean and SD estimated on all 2,551 developed tiles (797 floor tiles retained)
     and prints the resulting equations (coefficients and intercepts), so Equation 1 and 2 can be
     checked and, if needed, corrected.
  2. Recomputes tercile thresholds on the training set only and counts how many training and
     test tiles would change label (with Cohen's kappa). If few labels change, the leakage from
     the test set into preprocessing is shown to be negligible; if many change, retraining is needed.
  3. Exclusion sensitivity: terciles and the 3x3 label table when the 797 floor tiles are retained,
     plus how many of the 1,754 retained tiles would change label.
  4. Correlations (W vs F, density vs W, density vs F) under each population.

Input
  A single CSV with one row per developed tile (all 2,551, including the 797 excluded ones).
  Edit COLS below to match your column names.

Run
  python r1_preprocessing_audit.py tiles.csv --out results_r1
Send back
  the whole results_r1 folder.
"""
import argparse
import json
import os

import numpy as np
import pandas as pd
from scipy import stats

# ---------------------------------------------------------------- CONFIG
COLS = {
    "tile_id": "tile_id",
    "intersections": "intersection_density",   # nodes with 3+ segments per km2
    "amenities": "amenity_density",            # amenities per km2
    "entropy": "landuse_entropy",              # normalized 0..1
    "permeable": "permeable_ratio",            # (green + water) / tile area
    "setback": "setback_m",                    # mean building-to-water distance, capped at 1000
    "coverage": "building_coverage",           # building footprint / tile area
    "excluded": "excluded",                    # True for the 797 floor tiles
    "split": "split",                          # "train" / "test" (blank for excluded tiles)
    "walk_label": "walk_label",                # labels actually used in training (low/med/high)
    "walk_score": "walk_score",                # stored walkability composite (optional)
    "flood_score": "flood_score",              # stored flood composite (optional)
    "flood_label": "flood_label",
}
# Standardization the paper reports. Fill in from your code if these differ.
PUBLISHED = {
    "intersections": {"sd": 29.97, "mean": None},
    "amenities": {"sd": 61.09, "mean": None},
    "entropy": {"sd": 0.112, "mean": None},
    "permeable": {"sd": 0.327, "mean": None},
    "setback": {"sd": 323.1, "mean": None},
}
PUBLISHED_EQ = {
    "W": {"intersections": 0.0111231, "amenities": 0.00545628, "entropy": 2.98158, "intercept": -0.30415},
    "F": {"permeable": 1.53122, "setback": 0.00154735, "intercept": -1.52288},
}
LABELS = ["low", "med", "high"]
W_PARTS = ["intersections", "amenities", "entropy"]
F_PARTS = ["permeable", "setback"]
# ------------------------------------------------------------------------


def to_bool(s):
    if s.dtype == bool:
        return s
    return s.astype(str).str.strip().str.lower().isin(["1", "true", "yes", "y", "t"])


def fit_standardization(df, parts):
    return {p: {"mean": float(df[p].mean()), "sd": float(df[p].std(ddof=1))} for p in parts}


def composite(df, parts, stdz):
    z = [(df[p] - stdz[p]["mean"]) / stdz[p]["sd"] for p in parts]
    return sum(z) / len(parts)


def equation(parts, stdz):
    k = len(parts)
    coef = {p: 1.0 / (k * stdz[p]["sd"]) for p in parts}
    intercept = -sum(stdz[p]["mean"] / stdz[p]["sd"] for p in parts) / k
    return {**{p: round(c, 8) for p, c in coef.items()}, "intercept": round(intercept, 5)}


def terciles(x):
    return [float(np.quantile(x, 1 / 3)), float(np.quantile(x, 2 / 3))]


def label(x, cuts):
    return pd.cut(x, [-np.inf, cuts[0], cuts[1], np.inf], labels=LABELS).astype(str)


def kappa(a, b):
    a, b = np.asarray(a), np.asarray(b)
    po = np.mean(a == b)
    pe = sum(np.mean(a == c) * np.mean(b == c) for c in LABELS)
    return float((po - pe) / (1 - pe)) if pe < 1 else 1.0


def crosstab(w, f):
    t = pd.crosstab(pd.Categorical(w, LABELS), pd.Categorical(f, LABELS))
    t.index, t.columns = [f"walk_{c}" for c in LABELS], [f"flood_{c}" for c in LABELS]
    return t


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("tiles_csv")
    ap.add_argument("--out", default="results_r1")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)

    raw = pd.read_csv(args.tiles_csv)
    df = pd.DataFrame({k: raw[v] for k, v in COLS.items() if v in raw.columns})
    missing = [v for k, v in COLS.items() if v not in raw.columns]
    if missing:
        print("WARNING: columns not found (edit COLS):", missing)
    global F_PARTS
    F_PARTS = [p for p in F_PARTS if p in df.columns]
    if len(F_PARTS) < 2:
        print("NOTE: flood components not both present; flood standardization checks use the stored score only")
    df["excluded"] = to_bool(df["excluded"])
    kept = df[~df["excluded"]].copy()
    train = kept[kept["split"].astype(str).str.lower() == "train"]
    test = kept[kept["split"].astype(str).str.lower() == "test"]
    report = {"n_all_developed": len(df), "n_kept": len(kept), "n_train": len(train), "n_test": len(test)}

    # 0. Recover the composite actually used, by exact linear regression of the stored score
    rec = {}
    for score, parts in [("walk_score", W_PARTS), ("flood_score", F_PARTS)]:
        if score in raw.columns and parts:
            X = np.column_stack([df[p].values for p in parts] + [np.ones(len(df))])
            beta, res, *_ = np.linalg.lstsq(X, raw[score].values, rcond=None)
            fit = X @ beta
            rec[score] = {**{p: float(b) for p, b in zip(parts, beta[:-1])}, "intercept": float(beta[-1]),
                          "max_abs_residual": float(np.max(np.abs(fit - raw[score].values))),
                          "implied_sd": {p: float(1 / (len(parts) * b)) if b else None for p, b in zip(parts, beta[:-1])}}
    report["recovered_composites"] = rec

    # 1. Equations under each standardization population
    pops = {"training_only": train, "kept_1754": kept, "all_developed_2551": df}
    eqs = {}
    for name, pop in pops.items():
        sw, sf = fit_standardization(pop, W_PARTS), fit_standardization(pop, F_PARTS)
        eqs[name] = {"W_stats": sw, "F_stats": sf, "W_eq": equation(W_PARTS, sw),
                     "F_eq": equation(F_PARTS, sf) if F_PARTS else None}
    # Check which population reproduces the published equation
    pub_w = sum(PUBLISHED_EQ["W"][p] * kept[p] for p in W_PARTS) + PUBLISHED_EQ["W"]["intercept"]
    pub_f = (sum(PUBLISHED_EQ["F"][p] * kept[p] for p in F_PARTS) + PUBLISHED_EQ["F"]["intercept"]) if len(F_PARTS) == 2 else pd.Series([np.nan])
    eqs["published_equation_check"] = {
        "W_published_mean_over_kept": float(pub_w.mean()),
        "F_published_mean_over_kept": float(pub_f.mean()),
        "note": "If W_eq/F_eq of one population above matches PUBLISHED_EQ, that population was used.",
    }
    report["equations"] = eqs

    # 2. Training-only terciles vs labels used
    sw_tr, sf_tr = eqs["training_only"]["W_stats"], eqs["training_only"]["F_stats"]
    kept["W_tr"] = composite(kept, W_PARTS, sw_tr)
    kept["F_tr"] = composite(kept, F_PARTS, sf_tr) if len(F_PARTS) == 2 else raw.loc[kept.index, "flood_score"]
    tr_mask = kept["split"].astype(str).str.lower() == "train"
    cw, cf = terciles(kept.loc[tr_mask, "W_tr"]), terciles(kept.loc[tr_mask, "F_tr"])
    kept["walk_tr"], kept["flood_tr"] = label(kept["W_tr"], cw), label(kept["F_tr"], cf)
    have_labels = "walk_label" in kept and "flood_label" in kept
    leak = {"train_only_cuts_W": cw, "train_only_cuts_F": cf}
    if have_labels:
        for part, m in [("train", tr_mask), ("test", ~tr_mask)]:
            sub = kept[m]
            w_change = int((sub["walk_tr"] != sub["walk_label"].astype(str)).sum())
            f_change = int((sub["flood_tr"] != sub["flood_label"].astype(str)).sum())
            either = int(((sub["walk_tr"] != sub["walk_label"].astype(str)) |
                          (sub["flood_tr"] != sub["flood_label"].astype(str))).sum())
            leak[part] = {
                "n": int(len(sub)),
                "walk_label_changes": w_change, "flood_label_changes": f_change,
                "either_label_changes": either, "pct_either": round(100 * either / max(len(sub), 1), 2),
                "kappa_walk": round(kappa(sub["walk_tr"], sub["walk_label"].astype(str)), 4),
                "kappa_flood": round(kappa(sub["flood_tr"], sub["flood_label"].astype(str)), 4),
            }
        crosstab(kept.loc[~tr_mask, "walk_tr"], kept.loc[~tr_mask, "flood_tr"]).to_csv(
            os.path.join(args.out, "test_label_table_train_only_thresholds.csv"))
    report["leakage_check"] = leak

    # 3. Exclusion sensitivity: retain the 797 floor tiles
    sw_all, sf_all = eqs["all_developed_2551"]["W_stats"], eqs["all_developed_2551"]["F_stats"]
    df["W_all"] = composite(df, W_PARTS, sw_all)
    df["F_all"] = composite(df, F_PARTS, sf_all) if len(F_PARTS) == 2 else raw["flood_score"]
    cw_all, cf_all = terciles(df["W_all"]), terciles(df["F_all"])
    df["walk_all"], df["flood_all"] = label(df["W_all"], cw_all), label(df["F_all"], cf_all)
    tab_all = crosstab(df["walk_all"], df["flood_all"])
    tab_all.to_csv(os.path.join(args.out, "label_table_retaining_797.csv"))
    floor_W = df.loc[df["excluded"], "W_all"]
    sens = {
        "cuts_W_retaining_797": cw_all, "cuts_F_retaining_797": cf_all,
        "share_of_all_tiles_tied_at_W_minimum": round(float((df["W_all"] <= df["W_all"].min() + 1e-9).mean()), 4),
        "floor_tiles_W_unique_values": int(floor_W.round(9).nunique()),
        "floor_tiles_labeled": df.loc[df["excluded"], "walk_all"].value_counts().to_dict(),
        "high_high_cell_retaining_797": int(tab_all.loc["walk_high", "flood_high"]),
        "smallest_cell_retaining_797": int(tab_all.values.min()),
    }
    if have_labels:
        merged = df.loc[~df["excluded"]]  # same row order as `kept`
        same_w = (merged["walk_all"].values == kept["walk_label"].astype(str).values)
        same_f = (merged["flood_all"].values == kept["flood_label"].astype(str).values)
        sens["kept_tiles_changing_walk_label"] = int((~same_w).sum())
        sens["kept_tiles_changing_flood_label"] = int((~same_f).sum())
    report["exclusion_sensitivity"] = sens

    # 4. Correlations under each population
    corr = {}
    for name, d, wcol, fcol in [("kept_train_only_std", kept, "W_tr", "F_tr"),
                                ("all_2551", df, "W_all", "F_all")]:
        corr[name] = {
            "pearson_W_F": round(float(stats.pearsonr(d[wcol], d[fcol])[0]), 4),
            "spearman_W_F": round(float(stats.spearmanr(d[wcol], d[fcol])[0]), 4),
            "pearson_cov_W": round(float(stats.pearsonr(d["coverage"], d[wcol])[0]), 4),
            "pearson_cov_F": round(float(stats.pearsonr(d["coverage"], d[fcol])[0]), 4),
        }
    report["correlations"] = corr
    if "setback" in kept:
        report["setback_cap_share_kept"] = round(float((kept["setback"] >= 1000 - 1e-6).mean()), 4)

    with open(os.path.join(args.out, "r1_report.json"), "w") as fh:
        json.dump(report, fh, indent=2)
    print(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/r2_statistics.py
"""
R2. Repeated-measures statistics, spatial block-bootstrap uncertainty, effect sizes, and
multiple-comparison control (reviewer comment 7, and supporting comments 2 and 8).

CPU only. Uses the existing sweep outputs; no retraining.

What it does
  1. Linear mixed model per outcome (green, mix, building):
        outcome ~ C(walk) * C(flood) + (1 | canvas)
     with likelihood-ratio tests for the walkability effect, the flood effect, and their
     interaction, plus partial eta-squared from a repeated-measures ANOVA (balanced design).
  2. Spatial block bootstrap: whole 2 km superblocks of held-out canvases are resampled with
     replacement (default 5,000 draws), so spatial autocorrelation between neighbouring canvases
     is respected. Gives 95% CIs for every cell mean (Table 7), every marginal mean (Table 6),
     adjacent differences, the green fold change, and the high-walk corner cost (absolute and %).
  3. Paired adjacent-step tests on canvas-level marginal means (Wilcoxon signed-rank), Holm
     correction across the whole family, and paired effect sizes (Cohen's dz).
  4. Operational criteria used in the revised text:
        monotonic        = both adjacent marginal steps have Holm-adjusted p < 0.05 and
                           block-bootstrap 95% CIs that exclude zero, in the same direction
        indistinguishable= adjacent step 95% CI includes zero AND |step| < EQUIV_MARGIN (TOST-style)
        largely independent = the cross factor explains < 1% of within-canvas variance
                           (partial eta-squared) while the own factor explains more than 10x that
  5. Optional: unconditional-vs-conditional paired comparison and the ambiguity analysis with
     block-bootstrap CIs, if baseline.csv is supplied.

Inputs
  sweep.csv     long format, one row per (canvas, walk label, flood label): 269 x 9 = 2,421 rows
  canvases.csv  canvas_id -> superblock_id (and optionally ambiguity score)
  baseline.csv  (optional) canvas_id, green, mix, building for the unconditional model
Edit COLS to match your column names.

Run
  python r2_statistics.py sweep.csv canvases.csv [--baseline baseline.csv] --out results_r2
"""
import argparse
import json
import os
import warnings

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------- CONFIG
COLS = {
    "canvas": "canvas_id",
    "walk": "walk_label",
    "flood": "flood_label",
    "green": "green",
    "mix": "mix",
    "building": "building",
    "superblock": "superblock_id",
    "ambiguity": "ambiguity",
}
LABELS = ["low", "med", "high"]
OUTCOMES = ["green", "mix", "building"]
N_BOOT = 5000
SEED = 20260713
EQUIV_MARGIN = {"green": 0.02, "mix": 0.02, "building": 0.01}
NEAR_EMPTY = 0.02
# ------------------------------------------------------------------------


def norm_label(s):
    m = {"low": "low", "l": "low", "0": "low", "med": "med", "medium": "med", "m": "med", "1": "med",
         "high": "high", "h": "high", "2": "high"}
    return s.astype(str).str.strip().str.lower().map(m)


def holm(pvals):
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    m = len(p)
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p[idx])
        adj[idx] = min(1.0, running)
    return adj


def cell_table(d, y):
    return d.pivot_table(index="walk", columns="flood", values=y, aggfunc="mean").reindex(index=LABELS, columns=LABELS)


def summaries(d):
    """All quantities that get bootstrap CIs, from a long-format frame."""
    out = {}
    for y in OUTCOMES:
        cells = cell_table(d, y)
        for w in LABELS:
            for f in LABELS:
                out[f"{y}|cell|walk={w}|flood={f}"] = cells.loc[w, f]
        for fac in ["walk", "flood"]:
            marg = d.groupby(fac)[y].mean().reindex(LABELS)
            for lab in LABELS:
                out[f"{y}|marg|{fac}={lab}"] = marg[lab]
            out[f"{y}|step|{fac}:low->med"] = marg["med"] - marg["low"]
            out[f"{y}|step|{fac}:med->high"] = marg["high"] - marg["med"]
            out[f"{y}|step|{fac}:low->high"] = marg["high"] - marg["low"]
    g = d.groupby("flood")["green"].mean().reindex(LABELS)
    out["green|fold|flood high/low"] = g["high"] / g["low"]
    b = cell_table(d, "building")
    out["building|corner_cost_abs|(high,low)-(high,high)"] = b.loc["high", "low"] - b.loc["high", "high"]
    out["building|corner_cost_pct"] = 100 * (b.loc["high", "low"] - b.loc["high", "high"]) / b.loc["high", "low"]
    m = cell_table(d, "mix")
    out["mix|corner_minus_max"] = m.loc["high", "high"] - np.nanmax(m.values)
    return out


def block_bootstrap(d, blocks, rng, n_boot):
    groups = {b: g for b, g in d.groupby("superblock")}
    keys = list(groups)
    point = summaries(d)
    draws = {k: [] for k in point}
    for _ in range(n_boot):
        pick = rng.choice(len(keys), size=len(keys), replace=True)
        parts = []
        for j, i in enumerate(pick):
            g = groups[keys[i]].copy()
            g["canvas"] = g["canvas"].astype(str) + f"_b{j}"
            parts.append(g)
        s = summaries(pd.concat(parts, ignore_index=True))
        for k, v in s.items():
            draws[k].append(v)
    rows = []
    for k, v in point.items():
        arr = np.asarray(draws[k], float)
        lo, hi = np.nanpercentile(arr, [2.5, 97.5])
        rows.append({"quantity": k, "estimate": v, "ci_low": lo, "ci_high": hi})
    return pd.DataFrame(rows)


def mixed_models(d):
    import statsmodels.formula.api as smf
    from statsmodels.stats.anova import AnovaRM
    res = {}
    for y in OUTCOMES:
        full = smf.mixedlm(f"{y} ~ C(walk, Treatment('low')) * C(flood, Treatment('low'))", d,
                           groups=d["canvas"]).fit(reml=False)
        add = smf.mixedlm(f"{y} ~ C(walk, Treatment('low')) + C(flood, Treatment('low'))", d,
                          groups=d["canvas"]).fit(reml=False)
        no_w = smf.mixedlm(f"{y} ~ C(flood, Treatment('low'))", d, groups=d["canvas"]).fit(reml=False)
        no_f = smf.mixedlm(f"{y} ~ C(walk, Treatment('low'))", d, groups=d["canvas"]).fit(reml=False)

        def lrt(big, small, df):
            stat = 2 * (big.llf - small.llf)
            return {"chi2": round(float(stat), 3), "df": df, "p": float(stats.chi2.sf(stat, df))}

        aov = AnovaRM(d, depvar=y, subject="canvas", within=["walk", "flood"]).fit().anova_table
        eta = {}
        # partial eta^2 = F*df1 / (F*df1 + df2)
        for term in aov.index:
            F, df1, df2 = aov.loc[term, "F Value"], aov.loc[term, "Num DF"], aov.loc[term, "Den DF"]
            eta[term] = round(float(F * df1 / (F * df1 + df2)), 5)
        res[y] = {
            "LRT_walk": lrt(add, no_w, 2),
            "LRT_flood": lrt(add, no_f, 2),
            "LRT_interaction": lrt(full, add, 4),
            "partial_eta_sq": eta,
            "rmANOVA": {t: {"F": round(float(aov.loc[t, "F Value"]), 3), "p": float(aov.loc[t, "Pr > F"])}
                        for t in aov.index},
            "canvas_random_intercept_var": float(full.cov_re.iloc[0, 0]),
            "residual_var": float(full.scale),
        }
    return res


def paired_steps(d):
    rows = []
    for y in OUTCOMES:
        for fac in ["walk", "flood"]:
            cm = d.groupby(["canvas", fac])[y].mean().unstack().reindex(columns=LABELS)
            for a, b in [("low", "med"), ("med", "high"), ("low", "high")]:
                diff = cm[b] - cm[a]
                p = stats.wilcoxon(diff).pvalue if np.any(diff != 0) else 1.0
                dz = diff.mean() / diff.std(ddof=1) if diff.std(ddof=1) > 0 else np.nan
                rows.append({"outcome": y, "factor": fac, "step": f"{a}->{b}", "mean_diff": diff.mean(),
                             "cohens_dz": dz, "wilcoxon_p": p, "n_canvases": int(diff.notna().sum())})
    t = pd.DataFrame(rows)
    t["holm_p"] = holm(t["wilcoxon_p"])
    return t


def criteria(steps, boot, mm):
    crit = {}
    b = boot.set_index("quantity")
    for y in OUTCOMES:
        for fac in ["walk", "flood"]:
            s = steps[(steps.outcome == y) & (steps.factor == fac)].set_index("step")
            ci = [(b.loc[f"{y}|step|{fac}:{k}", "ci_low"], b.loc[f"{y}|step|{fac}:{k}", "ci_high"])
                  for k in ["low->med", "med->high"]]
            sig = [s.loc[k, "holm_p"] < 0.05 for k in ["low->med", "med->high"]]
            excl = [lo > 0 or hi < 0 for lo, hi in ci]
            signs = [np.sign(s.loc[k, "mean_diff"]) for k in ["low->med", "med->high"]]
            mono = all(sig) and all(excl) and signs[0] == signs[1]
            indist = {k: bool((lo <= 0 <= hi) and abs(s.loc[k, "mean_diff"]) < EQUIV_MARGIN[y])
                      for k, (lo, hi) in zip(["low->med", "med->high"], ci)}
            crit[f"{y} by {fac}"] = {"monotonic": bool(mono), "indistinguishable_steps": indist}
    for y, own, cross in [("green", "flood", "walk"), ("mix", "walk", "flood")]:
        e = mm[y]["partial_eta_sq"]
        crit[f"{y} independence"] = {"own_partial_eta_sq": e.get(own), "cross_partial_eta_sq": e.get(cross),
                                     "interaction_partial_eta_sq": e.get("walk:flood"),
                                     "largely_independent": bool(e.get(cross, 1) < 0.01 and
                                                                 e.get(own, 0) > 10 * e.get(cross, 1))}
    return crit


def baseline_analysis(d, base, canv, rng, n_boot):
    cond = d.groupby("canvas")[OUTCOMES].mean()
    base = base.set_index("canvas")[OUTCOMES]
    j = cond.join(base, lsuffix="_cond", rsuffix="_unc", how="inner").join(canv.set_index("canvas"))
    j["diff_building"] = j["building_unc"] - j["building_cond"]
    j["diff_green"] = j["green_unc"] - j["green_cond"]
    out = {
        "n": int(len(j)),
        "share_unc_builds_less": float((j["diff_building"] < 0).mean()),
        "mean_diff_building": float(j["diff_building"].mean()),
        "wilcoxon_building_p": float(stats.wilcoxon(j["diff_building"]).pvalue),
        "cohens_dz_building": float(j["diff_building"].mean() / j["diff_building"].std(ddof=1)),
        "mean_diff_green": float(j["diff_green"].mean()),
        "wilcoxon_green_p": float(stats.wilcoxon(j["diff_green"]).pvalue),
        "near_empty_unc": int((j["building_unc"] < NEAR_EMPTY).sum()),
        "near_empty_cond_mean": int((j["building_cond"] < NEAR_EMPTY).sum()),
    }
    out["holm_p"] = dict(zip(["building", "green"], holm([out["wilcoxon_building_p"], out["wilcoxon_green_p"]])))
    has_amb = "ambiguity" in j and j["ambiguity"].notna().any()
    if has_amb:
        rho, p = stats.spearmanr(j["ambiguity"], j["diff_building"])
        out["spearman_ambiguity_vs_deficit"] = {"rho": float(rho), "p": float(p)}
    # block bootstrap over superblocks
    groups = {b: g for b, g in j.groupby("superblock")}
    keys = list(groups)
    draws = {"mean_diff_building": [], "mean_diff_green": [], "share_less": [], "rho": []}
    for _ in range(n_boot):
        pick = rng.choice(len(keys), size=len(keys), replace=True)
        s = pd.concat([groups[keys[i]] for i in pick])
        draws["mean_diff_building"].append(s["diff_building"].mean())
        draws["mean_diff_green"].append(s["diff_green"].mean())
        draws["share_less"].append((s["diff_building"] < 0).mean())
        if has_amb:
            draws["rho"].append(stats.spearmanr(s["ambiguity"], s["diff_building"])[0])
    out["block_bootstrap_95ci"] = {k: [float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))]
                                   for k, v in draws.items() if len(v)}
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("sweep_csv")
    ap.add_argument("canvases_csv")
    ap.add_argument("--baseline")
    ap.add_argument("--boot", type=int, default=N_BOOT)
    ap.add_argument("--out", default="results_r2")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    rng = np.random.default_rng(SEED)

    sw = pd.read_csv(args.sweep_csv)
    d = pd.DataFrame({k: sw[COLS[k]] for k in ["canvas", "walk", "flood"] + OUTCOMES})
    d["walk"], d["flood"] = norm_label(d["walk"]), norm_label(d["flood"])
    cv = pd.read_csv(args.canvases_csv)
    canv = pd.DataFrame({"canvas": cv[COLS["canvas"]], "superblock": cv[COLS["superblock"]]})
    if COLS["ambiguity"] in cv:
        canv["ambiguity"] = cv[COLS["ambiguity"]]
    d = d.merge(canv[["canvas", "superblock"]], on="canvas", how="left")
    if d["superblock"].isna().any():
        raise SystemExit("Some canvases have no superblock_id; check canvases.csv")
    counts = d.groupby("canvas").size()
    print(f"{d['canvas'].nunique()} canvases, {d['superblock'].nunique()} superblocks, "
          f"rows per canvas: {counts.min()}-{counts.max()}")

    mm = mixed_models(d)
    steps = paired_steps(d)
    boot = block_bootstrap(d, "superblock", rng, args.boot)
    crit = criteria(steps, boot, mm)

    steps.to_csv(os.path.join(args.out, "paired_steps_holm.csv"), index=False)
    boot.to_csv(os.path.join(args.out, "block_bootstrap_ci.csv"), index=False)
    report = {"n_canvases": int(d["canvas"].nunique()), "n_superblocks": int(d["superblock"].nunique()),
              "n_boot": args.boot, "seed": SEED, "mixed_models": mm, "criteria": crit}
    if args.baseline:
        bs = pd.read_csv(args.baseline)
        base = pd.DataFrame({"canvas": bs[COLS["canvas"]], **{y: bs[COLS[y]] for y in OUTCOMES}})
        report["unconditional_vs_conditional"] = baseline_analysis(d, base, canv, rng, args.boot)
    with open(os.path.join(args.out, "r2_report.json"), "w") as fh:
        json.dump(report, fh, indent=2, default=float)
    print(json.dumps(report, indent=2, default=float))
    print(boot.to_string())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/r3_segmentation_metrics.py
"""
R3. Domain-relevant evaluation of generated layouts (reviewer comment 6).

CPU only. Works on PNG files the model has already produced (or that you regenerate on CPU).

What it does
  1. Decodes every image to class indices by nearest palette colour (Table 3 palette), and
     reports how far decoded pixels were from their nearest palette colour (decoding quality).
  2. Per-class intersection-over-union and Dice, mean IoU over the generated classes, pixel
     accuracy, and a confusion matrix (real class by generated class), for each variant:
       true labels, shuffled labels, unconditional model.
  3. A categorical per-tile disagreement rate (share of pixels whose class differs from the real
     tile). This replaces the RGB-based mean absolute error the reviewer questioned.
  4. Building boundary F-score with a 2 pixel tolerance (about 4 m).
  5. Building topology: number of connected building components per tile, generated vs real.
  6. Paired comparison of true vs shuffled labels per tile (Wilcoxon), so the label effect is
     shown on categorical metrics, not only on RGB error.

Roads and water are copied from the input canvas, so metrics are reported both over all
classes and over the generated classes only (background, green, four building classes).

Inputs (folders of PNGs named <canvas_id>.png, all 256x256)
  --real      real target layouts of the 269 held-out tiles
  --true      generations under each tile's true labels
  --shuffled  generations under shuffled labels (optional)
  --uncond    unconditional model generations (optional)

Run
  python r3_segmentation_metrics.py --real real/ --true gen_true/ --shuffled gen_shuf/ --uncond gen_unc/ --out results_r3
"""
import argparse
import glob
import json
import os

import numpy as np
from PIL import Image
from scipy import ndimage, stats

PALETTE = {  # Table 3
    "background": (0, 0, 0),
    "road": (140, 140, 140),
    "water": (0, 90, 255),
    "green": (0, 170, 0),
    "residential": (255, 190, 0),
    "commercial": (230, 30, 30),
    "industrial": (150, 60, 200),
    "unclassified": (240, 240, 240),
}
NAMES = list(PALETTE)
PAL = np.array([PALETTE[n] for n in NAMES], dtype=np.float32)
GENERATED = ["background", "green", "residential", "commercial", "industrial", "unclassified"]
BUILDING = ["residential", "commercial", "industrial", "unclassified"]
BUILDING_IDX = [NAMES.index(n) for n in BUILDING]


def decode(path):
    rgb = np.asarray(Image.open(path).convert("RGB"), dtype=np.float32)
    dist = ((rgb[..., None, :] - PAL[None, None]) ** 2).sum(-1)
    cls = dist.argmin(-1)
    return cls, np.sqrt(dist.min(-1))


def confusion(real, gen, k):
    return np.bincount(real.ravel() * k + gen.ravel(), minlength=k * k).reshape(k, k)


def iou_dice(cm):
    tp = np.diag(cm).astype(float)
    fp, fn = cm.sum(0) - tp, cm.sum(1) - tp
    with np.errstate(invalid="ignore", divide="ignore"):
        iou = tp / (tp + fp + fn)
        dice = 2 * tp / (2 * tp + fp + fn)
    return iou, dice


def boundary(mask):
    return mask & ~ndimage.binary_erosion(mask)


def boundary_f(real_b, gen_b, tol=2):
    rb, gb = boundary(real_b), boundary(gen_b)
    if rb.sum() == 0 and gb.sum() == 0:
        return np.nan
    dr = ndimage.distance_transform_edt(~rb) if rb.any() else np.full(rb.shape, np.inf)
    dg = ndimage.distance_transform_edt(~gb) if gb.any() else np.full(gb.shape, np.inf)
    prec = (dr[gb] <= tol).mean() if gb.any() else 0.0
    rec = (dg[rb] <= tol).mean() if rb.any() else 0.0
    return 0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec)


def evaluate(real_dir, gen_dir):
    k = len(NAMES)
    cm = np.zeros((k, k), np.int64)
    per_tile, dec_err = {}, []
    for rp in sorted(glob.glob(os.path.join(real_dir, "*.png"))):
        cid = os.path.splitext(os.path.basename(rp))[0]
        gp = os.path.join(gen_dir, cid + ".png")
        if not os.path.exists(gp):
            continue
        r, _ = decode(rp)
        g, err = decode(gp)
        dec_err.append(err.ravel())
        c = confusion(r, g, k)
        cm += c
        rb, gb = np.isin(r, BUILDING_IDX), np.isin(g, BUILDING_IDX)
        gen_mask = np.isin(r, [NAMES.index(n) for n in GENERATED])
        per_tile[cid] = {
            "disagreement_all": float((r != g).mean()),
            "disagreement_generated": float((r[gen_mask] != g[gen_mask]).mean()) if gen_mask.any() else np.nan,
            "building_iou": float((rb & gb).sum() / max((rb | gb).sum(), 1)),
            "boundary_f_2px": boundary_f(rb, gb),
            "components_real": int(ndimage.label(rb)[1]),
            "components_gen": int(ndimage.label(gb)[1]),
        }
    iou, dice = iou_dice(cm)
    gi = [NAMES.index(n) for n in GENERATED]
    dec = np.concatenate(dec_err) if dec_err else np.array([0.0])
    summary = {
        "n_tiles": len(per_tile),
        "pixel_accuracy_all": float(np.trace(cm) / cm.sum()),
        "pixel_accuracy_generated": float(cm[np.ix_(gi, gi)].trace() / cm[np.ix_(gi, gi)].sum()),
        "iou": {n: float(iou[i]) for i, n in enumerate(NAMES)},
        "dice": {n: float(dice[i]) for i, n in enumerate(NAMES)},
        "mean_iou_generated_classes": float(np.nanmean(iou[gi])),
        "mean_dice_generated_classes": float(np.nanmean(dice[gi])),
        "per_tile_mean": {m: float(np.nanmean([t[m] for t in per_tile.values()]))
                          for m in next(iter(per_tile.values()))} if per_tile else {},
        "decoding_distance_rgb": {"median": float(np.median(dec)), "p95": float(np.percentile(dec, 95)),
                                  "share_over_40": float((dec > 40).mean())},
        "confusion_matrix_rows_real_cols_generated": {"classes": NAMES, "counts": cm.tolist()},
    }
    comp = [(t["components_real"], t["components_gen"]) for t in per_tile.values()]
    if len(comp) > 2:
        cr, cg = np.array(comp).T
        summary["building_components"] = {"real_mean": float(cr.mean()), "gen_mean": float(cg.mean()),
                                          "spearman": float(stats.spearmanr(cr, cg)[0])}
    return summary, per_tile


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--real", required=True)
    ap.add_argument("--true", required=True)
    ap.add_argument("--shuffled")
    ap.add_argument("--uncond")
    ap.add_argument("--out", default="results_r3")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    report, tiles = {}, {}
    for name, folder in [("true_labels", a.true), ("shuffled_labels", a.shuffled), ("unconditional", a.uncond)]:
        if folder:
            report[name], tiles[name] = evaluate(a.real, folder)
            print(name, "mIoU(generated)=%.4f" % report[name]["mean_iou_generated_classes"])
    for other in ["shuffled_labels", "unconditional"]:
        if other in tiles:
            ids = sorted(set(tiles["true_labels"]) & set(tiles[other]))
            comp = {}
            for m in ["disagreement_generated", "building_iou", "boundary_f_2px"]:
                x = np.array([tiles["true_labels"][i][m] for i in ids], float)
                y = np.array([tiles[other][i][m] for i in ids], float)
                ok = ~np.isnan(x) & ~np.isnan(y)
                diff = x[ok] - y[ok]
                comp[m] = {"true_mean": float(x[ok].mean()), f"{other}_mean": float(y[ok].mean()),
                           "mean_diff": float(diff.mean()),
                           "wilcoxon_p": float(stats.wilcoxon(diff).pvalue) if np.any(diff != 0) else 1.0}
            report[f"paired_true_vs_{other}"] = comp
    with open(os.path.join(a.out, "r3_report.json"), "w") as fh:
        json.dump(report, fh, indent=2)
    with open(os.path.join(a.out, "r3_per_tile.json"), "w") as fh:
        json.dump(tiles, fh)
    print(json.dumps({k: v for k, v in report.items()}, indent=2)[:6000])


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/r4_partition_map.py
"""
R4. Map of the spatial train/test partition and the excluded tiles (reviewer comments 3 and 4).

Input: the same tiles.csv as R1, plus tile centroid coordinates (SVY21 metres) and superblock ids.
Output: partition_map.png (300 dpi) and partition_summary.json.

Run
  python r4_partition_map.py tiles.csv --out results_r4
"""
import argparse
import json
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Patch

COLS = {"x": "x", "y": "y", "excluded": "excluded", "split": "split", "superblock": "superblock_id"}
TILE = 500


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("tiles_csv")
    ap.add_argument("--out", default="results_r4")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    t = pd.read_csv(a.tiles_csv)
    exc = t[COLS["excluded"]].astype(str).str.lower().isin(["1", "true", "yes"])
    split = t[COLS["split"]].astype(str).str.lower()
    cat = pd.Series("train", index=t.index)
    cat[split == "test"] = "test"
    cat[exc] = "excluded"
    colors = {"train": "#9aa3ad", "test": "#1f6fb2", "excluded": "#d1495b"}
    fig, ax = plt.subplots(figsize=(10, 6.2), dpi=300)
    for c, col in colors.items():
        m = cat == c
        ax.scatter(t.loc[m, COLS["x"]], t.loc[m, COLS["y"]], s=4.2, marker="s", c=col, linewidths=0)
    ax.set_aspect("equal")
    ax.set_xlabel("Easting, SVY21 (m)")
    ax.set_ylabel("Northing, SVY21 (m)")
    ax.legend(handles=[Patch(color=colors["train"], label=f"Training ({(cat=='train').sum()})"),
                       Patch(color=colors["test"], label=f"Held-out test ({(cat=='test').sum()})"),
                       Patch(color=colors["excluded"], label=f"Excluded ({(cat=='excluded').sum()})")],
              loc="lower right", frameon=False, fontsize=8)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    fig.tight_layout()
    fig.savefig(os.path.join(a.out, "partition_map.png"))
    sb = t.loc[cat == "test", COLS["superblock"]]
    summary = {"counts": cat.value_counts().to_dict(), "n_test_superblocks": int(sb.nunique()),
               "test_superblock_ids": sorted(map(str, sb.unique())),
               "tiles_per_test_superblock": sb.value_counts().describe().to_dict()}
    with open(os.path.join(a.out, "partition_summary.json"), "w") as fh:
        json.dump(summary, fh, indent=2, default=str)
    print(json.dumps(summary, indent=2, default=str))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/r5_environment_report.py
"""
R5. Software and hardware report (reviewer comments 3 and 5).

Run this INSIDE the environment you trained in (for example a Colab notebook cell:
  !python r5_environment_report.py
or paste the body into a cell). If training was on Colab and that runtime is gone, run it in a
fresh Colab runtime and note the GPU type you remember using.
"""
import importlib
import json
import platform
import sys

PKGS = ["numpy", "pandas", "scipy", "torch", "torchvision", "osmnx", "geopandas", "shapely",
        "networkx", "rasterio", "PIL", "matplotlib", "pytorch_fid", "torchmetrics", "torch_fidelity",
        "cleanfid", "statsmodels", "sklearn"]
rep = {"python": sys.version.split()[0], "platform": platform.platform()}
for p in PKGS:
    try:
        m = importlib.import_module(p)
        rep[p] = getattr(m, "__version__", "installed")
    except Exception:
        rep[p] = None
try:
    import torch
    rep["cuda_available"] = torch.cuda.is_available()
    if torch.cuda.is_available():
        rep["gpu"] = torch.cuda.get_device_name(0)
        rep["cuda"] = torch.version.cuda
        rep["cudnn"] = torch.backends.cudnn.version()
except Exception:
    pass
print(json.dumps(rep, indent=2))


In [ ]:
# 5. Comments 3 and 4: preprocessing audit, exclusion sensitivity, partition map (Figure 1C)
!python /content/r1_preprocessing_audit.py /content/std_tiles.csv --out "{OUT}/results_r1" > "{OUT}/r1_log.txt" 2>&1; tail -n 40 "{OUT}/r1_log.txt"
!python /content/r4_partition_map.py /content/std_tiles.csv --out "{OUT}/results_r4"
from IPython.display import Image as IImage, display
display(IImage(f'{OUT}/results_r4/partition_map.png', width=700))

In [ ]:
# 6. Comment 7: mixed models, spatial block bootstrap, Holm correction (about 7 minutes)
!python /content/r2_statistics.py /content/std_sweep.csv /content/std_canvases.csv --baseline /content/std_baseline.csv --out "{OUT}/results_r2" > "{OUT}/r2_log.txt" 2>&1; tail -n 30 "{OUT}/r2_log.txt"

In [ ]:
# 7. Comment 6: IoU, Dice, confusion matrix, boundary F, topology on the saved generations
!python /content/r3_segmentation_metrics.py --real /content/fid_real --true /content/fid_cond --shuffled /content/fid_cond_shuffled --uncond /content/fid_base_zero --out "{OUT}/results_r3" > "{OUT}/r3_log.txt" 2>&1; head -n 40 "{OUT}/r3_log.txt"

In [ ]:
# 8. Figure 7 check: are the published panels from the retrained baseline in native mode?
import matplotlib; import matplotlib.pyplot as plt
PAL = np.array([(0,0,0),(140,140,140),(0,90,255),(0,170,0),(255,190,0),(230,30,30),(150,60,200),(240,240,240)], float)
def bcov(path):
    a = np.asarray(Image.open(path).convert('RGB'), float).reshape(-1, 1, 3)
    c = ((a - PAL[None]) ** 2).sum(-1).argmin(1)
    return np.isin(c, [4, 5, 6, 7]).mean()
ids = [299, 300, 301, 307, 334, 335]
bm = pd.read_csv(f'{EVAL}/baseline_native_metrics.csv').set_index('tile')
lines = []
for i in ids:
    p = f'/content/fid_base_zero/{i}.png'
    lines.append(f'tile {i}: building coverage from fid_base_zero = {bcov(p):.3f} | baseline_native_metrics.csv = {bm.loc[i, "building"]:.3f}' if os.path.exists(p) else f'tile {i}: no native image')
lines.append('sample images in pix2pix_runs: ' + str([f for f in os.listdir(RUNS) if f.endswith('.png')]))
print('\n'.join(lines)); open(f'{OUT}/figure7_check.txt', 'w').write('\n'.join(lines))
fig, ax = plt.subplots(2, 6, figsize=(18, 6.2))
for j, i in enumerate(ids):
    ax[0, j].imshow(Image.open(f'/content/dataset/test/input/tile_{i}.png')); ax[0, j].set_title(f'in {i}', fontsize=8)
    if os.path.exists(f'/content/fid_base_zero/{i}.png'): ax[1, j].imshow(Image.open(f'/content/fid_base_zero/{i}.png'))
for a in ax.flat: a.set_axis_off()
plt.tight_layout(); plt.savefig(f'{OUT}/figure7_native.png', dpi=150); plt.show()

In [ ]:
# 9. Comment 5: software report, then zip everything for upload
!python /content/r5_environment_report.py > "{OUT}/r5_environment.json"; cat "{OUT}/r5_environment.json"
!pip show pytorch-fid | head -2 >> "{OUT}/r5_environment.json"
z = shutil.make_archive(f'{MY}/revision_results_upload', 'zip', OUT)
print('\nDONE. Upload this file to the chat:', z)